<a href="https://colab.research.google.com/github/Nithya2405/ai-engineering-workbench/blob/main/04_agent_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For **Day 11**, we are tackling **Agent Memory (State Management)**.

Up until now, your agents have been "forgetful." If you ask a follow-up question, the agent has no idea what you just talked about. Today, we change that. In the industry, this is known as **Context Window Management** or **Session State**.

---

### 💻 Day 11: The "Remembering" Agent (Code)

To give an agent memory, we don't just send the current prompt; we send the **entire conversation history** back to the model every time.

In [ ]:
!pip install groq
import os
from groq import Groq
from google.colab import userdata
from IPython.display import clear_output, display, HTML

# 1. Setup
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

class AgentMemory:
    def __init__(self, window_size=5):
        self.history = [
            {"role": "system", "content": "You are a highly capable AI Engineer's Assistant. You remember previous details to provide context-aware support."}
        ]
        self.window_size = window_size

    def add_message(self, role, content):
        self.history.append({"role": role, "content": content})
        # Keep the system prompt + the last 'window_size' exchanges
        if len(self.history) > (self.window_size * 2) + 1:
            self.history = [self.history[0]] + self.history[-(self.window_size * 2):]

    def get_history(self):
        return self.history

# 2. Initialize Memory
memory = AgentMemory(window_size=6)

def chat():
    print("🚀 Day 11: Stateful Agent Active. Type 'quit' to exit.")
    while True:
        user_input = input("👤 You: ")
        if user_input.lower() == 'quit': break

        # Add user message to state
        memory.add_message("user", user_input)

        # Generate response using FULL state
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=memory.get_history()
        )

        response = completion.choices[0].message.content
        memory.add_message("assistant", response)

        # 10x UI: Clear and re-print for a clean chat feel
        clear_output(wait=True)
        for msg in memory.get_history():
            if msg['role'] == 'user':
                print(f"👤 You: {msg['content']}")
            elif msg['role'] == 'assistant':
                print(f"🤖 Assistant: {msg['content']}\n" + "-"*50)

# Run the interactive session
chat()

🚀 Day 11: Stateful Agent Active. Type 'quit' to exit.
